# 01 — Load & Preprocess PubMed (Colab)

Runs on Colab. Clones the project from GitHub, downloads a PubMed subset, cleans articles, sentence-tokenizes, and saves JSONL splits to your Google Drive so notebooks 02–05 can reuse them.

**Before running:** `Runtime → Change runtime type → T4 GPU`. (CPU also works for this notebook; GPU just speeds up the dataset cache.)

In [ ]:
# 1. Clone the project repo (gets the latest code from GitHub)
REPO_URL = 'https://github.com/Captain-Uchiha/scientific-paper-summarizer.git'
!rm -rf /content/scientific-summarizer
!git clone -q {REPO_URL} /content/scientific-summarizer
import sys, os
sys.path.insert(0, '/content/scientific-summarizer')
os.chdir('/content/scientific-summarizer')
print('Working dir:', os.getcwd())

Mounted at /content/drive


In [ ]:
# 2. Install dependencies
!pip -q install 'transformers>=4.40' 'datasets>=2.18' rouge-score evaluate nltk sentencepiece accelerate tqdm
import nltk; nltk.download('punkt', quiet=True); nltk.download('punkt_tab', quiet=True)
!nvidia-smi 2>/dev/null || echo 'No GPU (fine for notebook 01).'

In [ ]:
# 3. Mount Google Drive — used ONLY to persist processed dataset & model checkpoints
#    so notebooks 02-05 don't have to redo this work.
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

DATA_DIR = '/content/drive/MyDrive/scientific-summarizer-data'
PROJECT_DIR = DATA_DIR  # alias used by later cells
!mkdir -p {DATA_DIR}/dataset/processed {DATA_DIR}/models {DATA_DIR}/outputs
print('Persistent data dir:', DATA_DIR)

In [ ]:
# 4. Download PubMed subset
from datasets import load_dataset
N_TRAIN, N_VAL, N_TEST = 3000, 400, 400
train_raw = load_dataset('scientific_papers', 'pubmed', split=f'train[:{N_TRAIN}]', trust_remote_code=True)
val_raw   = load_dataset('scientific_papers', 'pubmed', split=f'validation[:{N_VAL}]', trust_remote_code=True)
test_raw  = load_dataset('scientific_papers', 'pubmed', split=f'test[:{N_TEST}]', trust_remote_code=True)
print(train_raw, val_raw, test_raw)

In [ ]:
# 5. Quick look at one record
rec = train_raw[0]
print('ARTICLE (first 800 chars):\n', rec['article'][:800])
print('\n--- ABSTRACT ---\n', rec['abstract'][:800])

In [ ]:
# 6. Clean + sentence-tokenize, filter, save as JSONL
import json, os
from tqdm import tqdm
from preprocessing.clean import clean_article, split_sentences, is_record_valid

OUT = f'{PROJECT_DIR}/dataset/processed'
os.makedirs(OUT, exist_ok=True)

def process_split(ds, out_path):
    kept = dropped = 0
    with open(out_path, 'w', encoding='utf-8') as f:
        for ex in tqdm(ds, desc=os.path.basename(out_path)):
            art = clean_article(ex['article'])
            abs_ = clean_article(ex['abstract'])
            if not is_record_valid(art, abs_):
                dropped += 1; continue
            rec = {
                'input_text': art,
                'target_text': abs_,
                'sentences': split_sentences(art),
            }
            f.write(json.dumps(rec) + '\n')
            kept += 1
    print(f'{out_path}: kept={kept} dropped={dropped}')

process_split(train_raw, f'{OUT}/train.jsonl')
process_split(val_raw,   f'{OUT}/val.jsonl')
process_split(test_raw,  f'{OUT}/test.jsonl')

In [ ]:
# 7. Quick stats
import json
import numpy as np

def length_stats(path):
    art_lens, abs_lens, sent_counts = [], [], []
    with open(path) as f:
        for line in f:
            r = json.loads(line)
            art_lens.append(len(r['input_text'].split()))
            abs_lens.append(len(r['target_text'].split()))
            sent_counts.append(len(r['sentences']))
    print(path)
    print('  article words   median/p95:', int(np.median(art_lens)), int(np.percentile(art_lens, 95)))
    print('  abstract words  median/p95:', int(np.median(abs_lens)), int(np.percentile(abs_lens, 95)))
    print('  sentences/paper median/p95:', int(np.median(sent_counts)), int(np.percentile(sent_counts, 95)))

length_stats(f'{OUT}/train.jsonl')
length_stats(f'{OUT}/val.jsonl')
length_stats(f'{OUT}/test.jsonl')